In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install docling torch transformers huggingface_hub sentencepiece protobuf tokenizers --upgrade

  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)


In [ ]:
import torch
import re
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSeq2SeqLM, PreTrainedTokenizerFast
from docling.document_converter import DocumentConverter

class TextCorrectionPipeline:
    def __init__(self, model_name: str = "protonx-models/protonx-legal-tc", max_tokens: int = 160):
        self.max_tokens = max_tokens

        # 1. Device check
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            print(f"Loading correction model on GPU: {torch.cuda.get_device_name(0)}...")
        else:
            self.device = torch.device("cpu")
            print("WARNING: CUDA not detected! Running on CPU...")

        # 2. Tải và nạp trực tiếp file tokenizer.json từ repo
        print("Loading fast tokenizer from tokenizer.json...")
        tok_file = hf_hub_download(repo_id=model_name, filename="tokenizer.json")
        self.tokenizer = PreTrainedTokenizerFast(
            tokenizer_file=tok_file,
            eos_token="</s>",
            unk_token="<unk>",
            pad_token="<pad>",
            model_max_length=self.max_tokens
        )

        # 3. Load model weights
        print(f"Loading model weights from {model_name}...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device.type == "cuda" else torch.float32
        ).to(self.device)
        self.model.eval()

    def correct_text_chunk(self, text: str) -> str:
        text = text.strip()
        if not text:
            return ""

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_tokens
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                num_beams=5,
                num_return_sequences=1,
                max_new_tokens=self.max_tokens,
                early_stopping=True,
                decoder_start_token_id=0,
                eos_token_id=1,
                pad_token_id=0
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def clean_paragraph(self, paragraph: str) -> str:
        heading_match = re.match(r"^(#+\s*)(.*)", paragraph)
        if heading_match:
            prefix = heading_match.group(1)
            content = heading_match.group(2)
            return f"{prefix}{self.correct_text_chunk(content)}"

        sentences = re.split(r"(?<=[.?!])\s+", paragraph)
        corrected_sentences = []
        for s in sentences:
            if len(s.strip()) > 3:
                corrected_sentences.append(self.correct_text_chunk(s))
            else:
                corrected_sentences.append(s)
        return " ".join(corrected_sentences)

def clean_markdown_artifacts(markdown_text: str) -> str:
    """Loại bỏ comment ảnh và các ký tự rác do OCR nhận diện nhầm."""
    text = re.sub(r"<!-- image -->", "", markdown_text)

    # Loại bỏ các dòng chỉ có 1-2 chữ cái đơn độc (như R, TH do watermark/đầu trang)
    lines = text.split("\n")
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if len(stripped) <= 2 and stripped.isalpha():
            continue
        cleaned_lines.append(line)

    cleaned_text = "\n".join(cleaned_lines)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)
    return cleaned_text

def fix_merged_executive_table(markdown_text: str) -> str:
    """
    Sửa lỗi ô bảng bị merge nhiều nhân sự vào cùng 1 hàng:
    Tách Mr./Ms. thành từng hàng riêng biệt tương ứng.
    """
    pattern = re.compile(
        r"(\|\s*Mr\.\s*[^|]*Truong\s*Ngoc\s*Lan.*?\s*General\s*Director\..*?)\s*\|\s*Appointed\s*on\..*?\n",
        re.DOTALL
    )

    fixed_rows = (
        "| Mr. Trường Ngọc Lân | General Director | Appointed on 19 October 2022 |\n"
        "| Mr. Do Ngoc Dinh | Deputy General Director | Appointed on 17 July 2017 |\n"
        "| Ms. Nguyen Thi Thu Hang | Chief Accountant | Appointed on 23 May 2018 |\n"
    )
    return pattern.sub(fixed_rows, markdown_text)

def process_document(source_path: str) -> str:
    # Bước 1: Parse PDF qua Docling
    print(f"1. Converting {source_path} via Docling...")
    converter = DocumentConverter()
    doc_result = converter.convert(source_path)
    raw_markdown = doc_result.document.export_to_markdown()

    # Bước 2: Dọn dẹp artifacts & định dạng bảng bị dồn
    print("2. Normalizing structure and cleaning artifacts...")
    cleaned_nd = clean_markdown_artifacts(raw_markdown)
    cleaned_nd = fix_merged_executive_table(cleaned_nd)

    # Bước 3: Sửa lỗi text bằng protonx-legal-tc
    print("3. Applying ProtonX Legal TC to text paragraphs...")
    corrector = TextCorrectionPipeline()

    blocks = cleaned_nd.split("\n\n")
    processed_blocks = []

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        if block.startswith("|") or "\n|" in block:
            processed_blocks.append(block)
        else:
            corrected = corrector.clean_paragraph(block)
            processed_blocks.append(corrected)

    final_result = "\n\n".join(processed_blocks)
    return final_result

print("Đã nạp toàn bộ pipeline thành công! Bạn có thể gọi hàm process_document(source) ở cell tiếp theo.")

Đã nạp toàn bộ pipeline thành công! Bạn có thể gọi hàm process_document(source) ở cell tiếp theo.


In [ ]:
import os

# Đường dẫn file bạn vừa tải lên Colab (ví dụ: 'test2.pdf' nếu nằm ở thư mục gốc)
source = "/content/VIX_Baocaotaichinh_2025_1.pdf"

if os.path.exists(source):
    cleaned_markdown = process_document(source)
    print("\n" + "="* 50 + " RESULT " + "=" * 50)
    print(cleaned_markdown[:1000]) # In thử 1000 ký tự đầu tiên

    # Lưu kết quả ra file markdown mới
    output_path = source.replace(".pdf", "_cleaned.md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned_markdown)
    print(f"\nSaved to: {output_path}")
else:
    print(f"File not found: {source}. Hãy kiểm tra lại tên file hoặc đường dẫn upload.")

1. Converting /content/VIX_Baocaotaichinh_2025_1.pdf via Docling...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1135.)
  return F.conv2d(
[WARNING] 2026-11-07 03:49:05,447 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-07 03:49:56,858 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-07 03:54:19,271 [RapidOCR] main.py:164: The text detection result is empty
[WARNING] 2026-11-07 03:58:27,737 [RapidOCR] main.py:164: The text detection result is empty


2. Normalizing structure and cleaning artifacts...
3. Applying ProtonX Legal TC to text paragraphs...
Loading fast tokenizer from tokenizer.json...
Loading model weights from protonx-models/protonx-legal-tc...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]